# Black-Scholes Options Pricing: Walkthrough

Step by step: price an option, read its Greeks, back out implied volatility, then test the model against real SPX quotes.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from options_pricing import black_scholes as bs, market
pd.set_option('display.precision', 4)

## 1. Price a call and a put
A textbook check (Hull): S = 42, K = 40, r = 10%, σ = 20%, six months. Expected: call 4.76, put 0.81.

In [ ]:
for kind in ('call', 'put'):
    print(kind, round(bs.price(S=42, K=40, T=0.5, r=0.10, sigma=0.20, kind=kind), 4))

## 2. Put-call parity
C − P must equal S − K·e^(−rT). If it didn't, there would be a riskless profit.

In [ ]:
c, p = bs.price(42, 40, 0.5, 0.10, 0.20, 'call'), bs.price(42, 40, 0.5, 0.10, 0.20, 'put')
print(c - p, 42 - 40 * np.exp(-0.10 * 0.5))

## 3. The Greeks
An at-the-money SPX-sized option: S = K = 7,200, 24 days, σ = 24%, r = 4.2%.

In [ ]:
pd.DataFrame({k: bs.greeks(7200, 7200, 24/365, 0.042, 0.24, k) for k in ('call', 'put')})

Reading it: a $1 index move changes the call by about $0.53 (delta); one vol point is worth about $7.34 (vega); each day costs the call holder about $4.09 (theta).

## 4. Implied volatility
Invert the model: which σ reproduces a given price?

In [ ]:
px = bs.price(7200, 7200, 24/365, 0.042, 0.24, 'call')
bs.implied_vol(px, 7200, 7200, 24/365, 0.042, 'call')

## 5. Real market: SPX options expiring 2026-12-18

In [ ]:
chain = market.clean_chain(market.load_chain('../data/' + market.SNAPSHOT_FILE))
T, r = market.year_fraction('2026-11-24', '2026-12-18'), 0.042
fwd = market.parity_forward(chain, market.SNAPSHOT_SPOT, T, r)
s_eff = fwd * np.exp(-r * T)
print(f'forward {fwd:,.2f}   parity-implied spot {s_eff:,.2f}   index close {market.SNAPSHOT_SPOT:,.2f}')
print('parity-implied r:', round(market.parity_rates(chain, market.SNAPSHOT_SPOT, T)[0], 4))

In [ ]:
smile = market.otm_smile(market.add_implied_vols(chain, s_eff, T, r, 0.0), fwd)
atm = market.atm_vol(smile, fwd)
print(f'ATM implied vol {atm:.2%}')
smile[['type', 'strike', 'mid', 'iv']].iloc[::4]

## 6. One volatility for every strike?
Black-Scholes says yes. The market says no.

In [ ]:
comp = market.compare_flat_vol(smile, s_eff, T, r, 0.0, atm)
comp[['type', 'strike', 'mid', 'bs_price', 'error_pct']].iloc[::4]

Low-strike puts trade far above the flat-vol price (crash protection is in demand), and high-strike calls trade far below it. That pattern is the **volatility skew**, and it is the main reason practitioners extend Black-Scholes with smile, stochastic-volatility and jump models.